# SciVer Full-Search v3 server notebook

Canonical thin operator notebook for a repository that was prepared manually before JupyterLab starts. Run the ordered stages below. All runtime configuration (`PINNED_COMMIT_SHA`, `SCIVER_DATASET_PATH`, `SCIVER_RUN_ID`, `API_URL`, `API_KEY`) is read from the repository `.env` file, so no interactive typing is required. Live stages run automatically when you execute their cells.

## 1. SETUP — locate repository, load .env, and validate the checkout

In [ ]:
import importlib
import os
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except Exception:  # pragma: no cover
    load_dotenv = None


def _locate_repository_root() -> Path:
    candidates = (Path.cwd(), *Path.cwd().parents)
    for candidate in candidates:
        if (candidate / '.git').exists() and (candidate / 'notebooks' / 'sciver_meta_harness.ipynb').is_file():
            return candidate.resolve()
    raise RuntimeError('Open JupyterLab from inside the manually prepared repository checkout')


def _required_setting(name: str) -> str:
    value = os.environ.get(name)
    if not value or not value.strip():
        raise RuntimeError(f'{name} is missing from .env / environment')
    return value.strip()


REPOSITORY_DIRECTORY = _locate_repository_root()
ENV_FILE = REPOSITORY_DIRECTORY / '.env'

if load_dotenv is not None and ENV_FILE.is_file():
    load_dotenv(dotenv_path=ENV_FILE, override=False)

PINNED_COMMIT_SHA = _required_setting('PINNED_COMMIT_SHA')
DATASET_PATH = Path(_required_setting('SCIVER_DATASET_PATH')).expanduser().resolve()
RUN_ID = _required_setting('SCIVER_RUN_ID')
API_URL = _required_setting('API_URL')
API_KEY = _required_setting('API_KEY')
if '\r' in API_KEY or '\n' in API_KEY:
    raise RuntimeError('API_KEY contains invalid characters')

PREPARATION_DIRECTORY = REPOSITORY_DIRECTORY / 'workspace' / 'meta_harness' / 'full_search_v3' / RUN_ID / 'preparation'
CONFIG_PATH = None

if str(REPOSITORY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIRECTORY))

from meta_harness.server_run import (
    EXPECTED_REPOSITORY_ORIGIN,
    ServerError,
    freeze_server_run_winner,
    inspect_server_run_final_status,
    inspect_server_run_smoke_receipt,
    inspect_server_run_status,
    preflight_server_run_final,
    preflight_search_run,
    prepare_run,
    run_meta_harness_smoke,
    start_or_resume_server_run_final,
    start_or_resume_search_run,
    validate_server_run_checkout,
)
from meta_harness.final_evaluation import (
    final_evaluation_completion_receipt_path,
    load_final_evaluation_completion_receipt,
)

checkout = validate_server_run_checkout(
    repository_root=REPOSITORY_DIRECTORY,
    pinned_commit_sha=PINNED_COMMIT_SHA,
    expected_origin_url=EXPECTED_REPOSITORY_ORIGIN,
)
required_imports = ('requests', 'PIL', 'meta_harness.server_run')
for module_name in required_imports:
    importlib.import_module(module_name)
if not DATASET_PATH.is_file():
    raise FileNotFoundError('Configured SciVer dataset path does not exist')
{
    'checkout': checkout,
    'env_file': str(ENV_FILE),
    'api_url_loaded': bool(API_URL),
    'api_key_loaded': bool(API_KEY),
    'dataset_present': True,
    'run_id': RUN_ID,
}

Environment creation and `python -m pip install -r requirements.txt` (including `python-dotenv`) happen before JupyterLab is launched. This notebook verifies dependency metadata and imports; it does not install packages or modify the checkout.

## 2. SETUP — prepare or verify the deterministic split

In [ ]:
prepared = prepare_run(
    dataset_path=DATASET_PATH,
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    preparation_directory=PREPARATION_DIRECTORY,
    config_path=CONFIG_PATH,
)
{
    'run_id': prepared['run_id'],
    'split_sha256': prepared['split_sha256'],
    'search': prepared['SEARCH'],
    'final': prepared['FINAL'],
    'sample_overlap_count': prepared['sample_overlap_count'],
    'paper_overlap_count': prepared['paper_overlap_count'],
}

## 3. OFFLINE_SMOKE — repository-owned request and run preflight

In [ ]:
offline_smoke = preflight_search_run(
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    search_safe_manifest_path=prepared['search_safe_manifest_path'],
    search_records_path=prepared['search_dataset_path'],
    source_commit=PINNED_COMMIT_SHA,
)
{
    key: offline_smoke[key]
    for key in ('protocol_id', 'run_id', 'resume', 'resume_identity_checked', 'config_sha256', 'split_sha256', 'search_membership_sha256', 'canonical_p0_prompt_sha256', 'solver', 'parser_version', 'offline_smoke', 'workload', 'checkpoints')
}

## 4. LIVE_SMOKE — authorized model-list preflight and one canonical P0 POST

Credentials come from `.env` (`API_URL`, `API_KEY`); no typing is required. This runs automatically.

In [ ]:
live_smoke = run_meta_harness_smoke(
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    search_safe_manifest_path=prepared['search_safe_manifest_path'],
    search_records_path=prepared['search_dataset_path'],
    authorize_smoke_execution=True,
    api_url=API_URL,
    api_key=API_KEY,
    source_commit=PINNED_COMMIT_SHA,
)
live_smoke

## 5. Smoke receipt validation

In [ ]:
try:
    smoke_receipt = inspect_server_run_smoke_receipt(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
except ServerError:
    smoke_receipt = {'status': 'missing_or_incompatible'}
smoke_receipt

## 6. FULL_SEARCH — auto-started (or resumed) SEARCH

Runs automatically; credentials are read from `.env`. No typing required.

In [ ]:
search_result = start_or_resume_search_run(
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    search_safe_manifest_path=prepared['search_safe_manifest_path'],
    search_records_path=prepared['search_dataset_path'],
    authorize_search_execution=True,
    api_url=API_URL,
    api_key=API_KEY,
    source_commit=PINNED_COMMIT_SHA,
)
search_result['search']

## 7. SEARCH status

In [ ]:
try:
    search_status = inspect_server_run_status(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
except ServerError:
    search_status = {'status': 'not_started'}
search_status

## 8. Freeze — immutable SEARCH-only winner

In [ ]:
frozen_winner = None
if search_status.get('status') in {'patience_stopped', 'max_stopped'}:
    frozen_winner = freeze_server_run_winner(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
    frozen_winner
else:
    print('SEARCH is not terminal; no winner was frozen and FINAL remains locked.')

## 9. FINAL preflight — offline and frozen-identity bound

In [ ]:
if frozen_winner is None:
    final_preflight = {'status': 'locked_until_search_freeze'}
else:
    final_preflight = preflight_server_run_final(
        repository_root=REPOSITORY_DIRECTORY,
        run_id=RUN_ID,
        dataset_path=DATASET_PATH,
        private_manifest_path=prepared['private_manifest_path'],
        search_safe_manifest_path=prepared['search_safe_manifest_path'],
    )
final_preflight

## 10. FINAL — auto-executed paired P0/P* run

Runs automatically when a winner is frozen and FINAL preflight succeeds. Credentials are read from `.env`; no typing required.

In [ ]:
if frozen_winner is None:
    raise RuntimeError('FINAL remains locked until a valid winner is frozen')
final_result = start_or_resume_server_run_final(
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    dataset_path=DATASET_PATH,
    private_manifest_path=prepared['private_manifest_path'],
    search_safe_manifest_path=prepared['search_safe_manifest_path'],
    authorize_final_execution=True,
    api_url=API_URL,
    api_key=API_KEY,
)
final_result['final']

## 11. Sanitized aggregate reporting

In [ ]:
try:
    final_status = inspect_server_run_final_status(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
except ServerError:
    final_status = {'status': 'not_started'}
{
    'run_id': RUN_ID,
    'search': search_status,
    'frozen_winner': frozen_winner,
    'final': final_status,
    'artifact_root': str(REPOSITORY_DIRECTORY / 'workspace' / 'meta_harness' / 'full_search_v3' / RUN_ID),
}

## 12. COMPARISON — baseline (original P0) vs optimized (P* after SEARCH) on FINAL

Loads the FINAL completion receipt and compares the original baseline prompt (`cot`, P0) against the optimized prompt selected after SEARCH iterations (`meta_cot`, P*). Renders a numeric table and a bar visualization with no additional plotting dependency.

In [ ]:
from IPython.display import HTML, display

comparison_data = {'status': 'no_final_receipt'}
final_receipt_path = final_evaluation_completion_receipt_path(REPOSITORY_DIRECTORY, RUN_ID)
if final_receipt_path.is_file():
    receipt = load_final_evaluation_completion_receipt(final_receipt_path)
    variants = {v['prompt_variant']: v for v in receipt['variants']}
    comparison_data = {
        'status': 'complete',
        'baseline': variants['cot'],
        'optimized': variants['meta_cot'],
    }

if comparison_data['status'] == 'complete':
    metric_fields = ('accuracy', 'macro_f1', 'parse_coverage')
    labels = {'accuracy': 'Accuracy', 'macro_f1': 'Macro-F1', 'parse_coverage': 'Parse coverage'}
    base = comparison_data['baseline'].get('metrics', {})
    opt = comparison_data['optimized'].get('metrics', {})

    rows = []
    for f in metric_fields:
        b, o = base.get(f), opt.get(f)
        if isinstance(b, (int, float)) and isinstance(o, (int, float)):
            rows.append((labels[f], float(b), float(o)))

    table_html = (
        '<table style="border-collapse:collapse;font-family:sans-serif;font-size:14px;">'
        '<tr style="background:#f0f0f0;">'
        '<th style="border:1px solid #ccc;padding:6px 12px;">Metric</th>'
        '<th style="border:1px solid #ccc;padding:6px 12px;">Baseline (P0, original)</th>'
        '<th style="border:1px solid #ccc;padding:6px 12px;">Optimized (P*, after SEARCH)</th>'
        '<th style="border:1px solid #ccc;padding:6px 12px;">Delta</th>'
        '</tr>'
    )
    max_val = max([v for _, b, o in rows for v in (b, o)] + [1e-6])
    bar_html = ''
    for label, b, o in rows:
        delta = o - b
        color = '#2e7d32' if delta >= 0 else '#c62828'
        table_html += (
            f'<tr><td style="border:1px solid #ccc;padding:6px 12px;">{label}</td>'
            f'<td style="border:1px solid #ccc;padding:6px 12px;text-align:right;">{b:.4f}</td>'
            f'<td style="border:1px solid #ccc;padding:6px 12px;text-align:right;">{o:.4f}</td>'
            f'<td style="border:1px solid #ccc;padding:6px 12px;text-align:right;color:{color};font-weight:600;">{delta:+.4f}</td>'
            '</tr>'
        )
        bw = 100.0 * b / max_val
        ow = 100.0 * o / max_val
        bar_html += (
            f'<div style="margin-bottom:14px;font-family:sans-serif;">'
            f'<div style="font-size:13px;font-weight:600;margin-bottom:4px;">{label}</div>'
            f'<div style="display:flex;align-items:center;gap:8px;">'
            f'<div style="width:160px;font-size:12px;color:#555;text-align:right;">Baseline</div>'
            f'<div style="background:#eceff1;border-radius:4px;height:20px;width:100%;">'
            f'<div style="background:#1e88e5;width:{bw:.1f}%;height:20px;border-radius:4px;"></div></div>'
            f'<div style="width:60px;font-size:12px;">{b:.3f}</div>'
            f'</div>'
            f'<div style="display:flex;align-items:center;gap:8px;margin-top:4px;">'
            f'<div style="width:160px;font-size:12px;color:#555;text-align:right;">Optimized</div>'
            f'<div style="background:#eceff1;border-radius:4px;height:20px;width:100%;">'
            f'<div style="background:#43a047;width:{ow:.1f}%;height:20px;border-radius:4px;"></div></div>'
            f'<div style="width:60px;font-size:12px;">{o:.3f}</div>'
            f'</div></div>'
        )
    table_html += '</table>'

    display(HTML(
        f'<h3 style="font-family:sans-serif;">FINAL comparison — run <code>{RUN_ID}</code></h3>'
        f'<p style="font-family:sans-serif;font-size:13px;">'
        f'Baseline (P0): candidate <code>{comparison_data["baseline"].get("candidate_id")}</code> · prompt <code>{comparison_data["baseline"].get("prompt_sha256", "")[:12]}…</code><br>'
        f'Optimized (P*): candidate <code>{comparison_data["optimized"].get("candidate_id")}</code> · prompt <code>{comparison_data["optimized"].get("prompt_sha256", "")[:12]}…</code>'
        f'</p>'
    ))
    display(HTML(table_html))
    display(HTML('<h3 style="font-family:sans-serif;">Visualization</h3>' + bar_html))
else:
    print('No completed FINAL receipt is available to compare.')

comparison_data